# Healthcare Lakehouse — Phase 2: CDC + Delta MERGE

We will extend the existing healthcare project with simulated Change Data Capture (CDC) and an **atomic Delta Lake MERGE**. Our original Bronze/Silver/Gold Parquet pipeline remains intact.

**Expected:** 10,000 claims → 100 inserts + 20 updates − 5 deletes → **10,095 current claims**.

The update events use a timestamp to choose the latest change per claim. Five older events intentionally test deduplication. Rerunning this batch is designed to leave the **same current-state data**.

All data is synthetic. This is a local Spark demonstration, **not** a live streaming CDC service or HIPAA-compliant infrastructure.

## Step 1 — Install compatible Spark + Delta Lake

Use a fresh Google Colab runtime to avoid conflicts with an already-created SparkSession. Colab's preinstalled `dataproc-spark-connect` package may warn that it expects Spark 4; this notebook uses Spark 3.5 and compatible Delta 3.3.

**After installing, if Colab says restart session, restart and resume at the next cell.**

In [1]:
%pip -q install "pyspark>=3.5.3,<3.6" "delta-spark>=3.3,<3.4"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.10 which is incompatible.


## Step 2 — Clone your GitHub repository and start Delta-enabled Spark

Delta Lake needs Spark SQL extensions and its data source configured *before* we create a SparkSession. The Delta helper adds the matching JVM dependencies.

In [2]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/content/healthcare-lakehouse-data-platform")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/mosesrathan19/healthcare-lakehouse-data-platform.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(PROJECT))
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F
builder = (SparkSession.builder.master("local[*]")
    .appName("HealthcareDeltaCDC")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark version:", spark.version)

Spark version: 3.5.10


## Step 3 — Generate the baseline

The existing `generate` and `build` functions create 10,000 synthetic claims and 10,000 valid encounters, plus the original Parquet Silver and Gold layers. We retain them for reference.

In [3]:
from src.generate import generate
from src.pipeline import build
from src.delta_cdc import synthetic_events, latest_events, run
OUTPUT = PROJECT / "outputs" / "healthcare_delta_demo"
if not (OUTPUT / "silver" / "claims").exists():
    print("Generating synthetic Bronze and Silver...")
    print(generate(spark, OUTPUT, "demo"))
    print(build(spark, OUTPUT))
base = spark.read.parquet(str(OUTPUT / "silver" / "claims"))
assert base.count() == 10000
print("Baseline claims:", base.count())

Generating synthetic Bronze and Silver...
{'patients': 1000, 'providers': 100, 'encounters': 10000, 'claims': 10000}
{'patients': 1000, 'providers': 100, 'encounters_in': 10000, 'encounters_valid': 10000, 'encounters_rejected': 0, 'claims_in': 10000, 'claims_valid': 10000, 'claims_rejected': 0}
Baseline claims: 10000


## Step 4 — Inspect the simulated CDC events

`I` means insert, `U` means update, and `D` means delete. The feed has 130 records, but only 125 latest changes after filtering 5 stale updates.

In [4]:
bronze = spark.read.parquet(str(OUTPUT / "bronze" / "claims"))
events = synthetic_events(spark, bronze)
events.groupBy("op").count().orderBy("op").show()
print("All events:", events.count())
effective = latest_events(events)
print("Latest events:", effective.count())
assert events.count() == 130
assert effective.count() == 125
effective.orderBy("claim_id").show(10, truncate=False)

+---+-----+
| op|count|
+---+-----+
|  D|    5|
|  I|  100|
|  U|   25|
+---+-----+

All events: 130
Latest events: 125
+---------------+---------------+------------+------------+------------+---+-------------------+
|claim_id       |encounter_id   |billed_cents|claim_status|service_date|op |event_version      |
+---------------+---------------+------------+------------+------------+---+-------------------+
|CLM000000000001|ENC000000000001|10500       |DENIED      |2025-01-01  |U  |2026-11-24 12:00:00|
|CLM000000000002|ENC000000000002|10550       |PAID        |2025-01-02  |U  |2026-11-24 12:00:00|
|CLM000000000003|ENC000000000003|10600       |PAID        |2025-01-03  |U  |2026-11-24 12:00:00|
|CLM000000000004|ENC000000000004|10650       |PAID        |2025-01-04  |U  |2026-11-24 12:00:00|
|CLM000000000005|ENC000000000005|10700       |PAID        |2025-01-05  |U  |2026-11-24 12:00:00|
|CLM000000000006|ENC000000000006|10750       |PAID        |2025-01-06  |U  |2026-11-24 12:00:00|
|CLM000

## Step 5 — Execute Delta MERGE

The `run` function initializes Delta from Silver if needed, chooses the latest event per claim, applies transactional updates/deletes/inserts, checks foreign keys, and writes a Gold claims-by-status table.

In [5]:
report = run(spark, OUTPUT)
print(report)
assert report["current_claims"] == 10095
delta_claims = spark.read.format("delta").load(str(OUTPUT / "delta" / "claims"))
delta_claims.groupBy("claim_status").count().show()
delta_claims.select("claim_id", "billed_cents", "event_version").orderBy("claim_id").show(5, False)

{'batch': 'demo-2026-11-24', 'baseline_claims': 10000, 'events': 130, 'effective_events': 125, 'current_claims': 10095, 'claims_before_this_run': 10000, 'expected_inserted': 100, 'expected_updated': 20, 'expected_deleted': 5}
+------------+-----+
|claim_status|count|
+------------+-----+
|      DENIED|  919|
|        PAID| 9176|
+------------+-----+

+---------------+------------+-------------------+
|claim_id       |billed_cents|event_version      |
+---------------+------------+-------------------+
|CLM000000000001|10500       |2026-11-24 12:00:00|
|CLM000000000002|10550       |2026-11-24 12:00:00|
|CLM000000000003|10600       |2026-11-24 12:00:00|
|CLM000000000004|10650       |2026-11-24 12:00:00|
|CLM000000000005|10700       |2026-11-24 12:00:00|
+---------------+------------+-------------------+
only showing top 5 rows



## Step 6 — Prove replay safety

Run the same batch again. The code only updates a matched row when the incoming event version is newer than the row's version. This prevents duplicate inserts and repeats of the same updates. It also ignores unmatched delete events.

**Important limitation:** Real multi-batch CDC must retain tombstones and per-key sequence information so late events cannot resurrect deleted records.

In [6]:
from src.delta_cdc import merge_claims
before_replay = spark.read.format("delta").load(str(OUTPUT / "delta" / "claims"))
# Collect a stable deterministic fingerprint across all rows without collecting them.
from pyspark.sql import functions as F
def fingerprint(df):
    columns = sorted(df.columns)
    row_hash = F.xxhash64(*[F.col(c) for c in columns])
    return df.select(row_hash.alias("h")).agg(
        F.count("*").alias("n"), F.sum("h").alias("sum_h")
    ).first()
before_sig = fingerprint(before_replay)
second = run(spark, OUTPUT)
after_sig = fingerprint(spark.read.format("delta").load(str(OUTPUT / "delta" / "claims")))
assert before_sig == after_sig
assert second["current_claims"] == 10095
print("Replay preserved row count and fingerprint:", second["current_claims"])

Replay preserved row count and fingerprint: 10095


## Step 7 — Query Gold and inspect Delta history

Delta stores table commits in a transaction log. Use history to see versions and MERGE operations. Our Gold Parquet output is derived from the **committed Delta claims**.

In [7]:
from delta.tables import DeltaTable
table = DeltaTable.forPath(spark, str(OUTPUT / "delta" / "claims"))
table.history(10).select("version", "operation", "timestamp").show(truncate=False)
spark.read.parquet(str(OUTPUT / "delta_gold" / "claims_by_status")).show()

+-------+---------+-----------------------+
|version|operation|timestamp              |
+-------+---------+-----------------------+
|2      |MERGE    |2026-11-24 17:38:26.693|
|1      |MERGE    |2026-11-24 17:37:52.17 |
|0      |WRITE    |2026-11-24 17:37:29.529|
+-------+---------+-----------------------+

+------------+------+------------+----------+
|claim_status|claims|billed_cents|paid_cents|
+------------+------+------------+----------+
|      DENIED|   919|    91111900|         0|
|        PAID|  9176|   910040100| 910040100|
+------------+------+------------+----------+



## What's next?

After verifying this notebook, we can add **persistent CDC checkpoints across multiple batches**, explicit tombstone handling, data quality expectations, **1-million-record performance benchmarks**, and orchestration with Airflow or Databricks Workflows.

Do not publish Colab's generated data to GitHub. Git tracks **code and documentation**, while large datasets live in Parquet/Delta storage.